In [ ]:
# we make ad-hoc plots of the spectra in the 20250804 paramfiles.
# mostly this is cross spectra with ACT, and simple knox covariances.
# we also do a basic calibration to theory

import numpy as np
import matplotlib.pyplot as plt
from pspy import pspy_utils, so_spectra, so_dict
from pspipe_utils import misc

d1119 = so_dict.so_dict()
d1119.read_from_file('/home/zatkins/projects/PSpipe_dev/repos/PSpipe/project/SO/pISO/paramfiles/tiger_dr6xdeep56_20251119.dict')

d1019 = so_dict.so_dict()
d1019.read_from_file('/home/zatkins/projects/PSpipe_dev/repos/PSpipe/project/SO/pISO/paramfiles/tiger_dr6xdeep56_20251119_but_20251019_maps.dict')

spectra = ["TT", "TE", "TB", "ET", "BT", "EE", "EB", "BE", "BB"]

bin_low, bin_hi, bin_cent, bin_size = pspy_utils.read_binning_file(d1119['binning_file'], d1119['lmax'])

l_th, cmb_th = so_spectra.read_ps(d1119['best_fits_dir'] + 'cmb.dat', spectra)
assert l_th[0] == 2, "l_th should start at 2"

In [ ]:
so_maps = ['i1_f090', 'i3_f090', 'i4_f090', 'i6_f090', 'i1_f150', 'i3_f150', 'i4_f150', 'i6_f150', 'c1_f220', 'i5_f220', 'c1_f280', 'i5_f280']
lmax_for_plot = 3000

Dl_avg_1119 = {so_map: {} for so_map in so_maps}
Dl_auto_1119 = {so_map: {} for so_map in so_maps}
Dl_avg_1019 = {so_map: {} for so_map in so_maps}
Dl_auto_1019 = {so_map: {} for so_map in so_maps}
for so_map in so_maps:
    Dl_avg_1119[so_map] = {spec: 0 for spec in spectra}
    for dr6_map in ('pa5_f090', 'pa5_f150', 'pa6_f090', 'pa6_f150'):
        lb, ps_dict = so_spectra.read_ps(d1119['spec_dir'] + f'Dl_dr6_{dr6_map}xso_{so_map}_cross.dat', spectra)
        for spec in spectra:
            Dl_avg_1119[so_map][spec] += ps_dict[spec]/4

    lb, Dl_auto_1119[so_map] = so_spectra.read_ps(d1119['spec_dir'] + f'Dl_so_{so_map}xso_{so_map}_cross.dat', spectra)

    Dl_avg_1019[so_map] = {spec: 0 for spec in spectra}
    for dr6_map in ('pa5_f090', 'pa5_f150', 'pa6_f090', 'pa6_f150'):
        lb, ps_dict = so_spectra.read_ps(d1019['spec_dir'] + f'Dl_dr6_{dr6_map}xso_{so_map}_cross.dat', spectra)
        for spec in spectra:
            Dl_avg_1019[so_map][spec] += ps_dict[spec]/4

    lb, Dl_auto_1019[so_map] = so_spectra.read_ps(d1019['spec_dir'] + f'Dl_so_{so_map}xso_{so_map}_cross.dat', spectra)

for spec in ['TT', 'TE', 'TB', 'EE', 'EB', 'BB']:
    for so_map in so_maps:
        if 'f280' in so_map:
            ylims = {
                'TT': (-1000, 7000),
                'TE': (-200, 200),
                'TB': (-150, 150),
                'EE': (-10, 60),
                'EB': (-50, 50),
                'BB': (-50, 50)
            }
        else:
            ylims = {
                'TT': (-1000, 7000),
                'TE': (-200, 200),
                'TB': (-50, 50),
                'EE': (-10, 60),
                'EB': (-10, 10),
                'BB': (-10, 10)
            } 

        fig, ax = plt.subplots(figsize=(12, 6))

        mask = lb <= lmax_for_plot + 1000
        mask_th = l_th <= lmax_for_plot + 1000

        ax.plot(lb[mask], Dl_auto_1119[so_map][spec][mask], marker='.', markerfacecolor='white', markersize=10, color='C0', label=f'{so_map} x {so_map} (1119: $-$, 1019: --)')
        ax.plot(lb[mask], Dl_avg_1119[so_map][spec][mask], marker='.', markerfacecolor='white', markersize=10, color='C1', label=f'avg(ACT) x {so_map} (1119: $-$, 1019: --)')
        if spec != spec[::-1]:
            ax.plot(lb[mask], Dl_avg_1119[so_map][spec[::-1]][mask], marker='.', markerfacecolor='white', markersize=10, color='C2', label=f'{so_map} x avg(ACT) (1119: $-$, 1019: --)')

        ax.plot(lb[mask], Dl_auto_1019[so_map][spec][mask], marker='.', markerfacecolor='white', markersize=10, color='C0', linestyle='--')
        ax.plot(lb[mask], Dl_avg_1019[so_map][spec][mask], marker='.', markerfacecolor='white', markersize=10, color='C1', linestyle='--')
        if spec != spec[::-1]:
            ax.plot(lb[mask], Dl_avg_1019[so_map][spec[::-1]][mask], marker='.', markerfacecolor='white', markersize=10, color='C2', linestyle='--')

        ax.plot(l_th[mask_th], cmb_th[spec][mask_th], color='k', alpha=0.5, linestyle='--', zorder=0, label='LCDM (CMB-only)')
        ax.set_title(f'{so_map} {spec}', fontsize=16)
        ax.set_xlim(0, lmax_for_plot)
        ax.set_ylim(ylims[spec])
        ax.set_xlabel(r'$\ell$', fontsize=16)
        ax.set_ylabel(fr'$D_\ell^{{{spec}}}\ \rm [\mu K^2]$', fontsize=16)
        ax.legend(fontsize=16)
        plt.show()

In [ ]:
so_maps = ['i1_f090', 'i3_f090', 'i4_f090', 'i6_f090', 'i1_f150', 'i3_f150', 'i4_f150', 'i6_f150', 'c1_f220', 'i5_f220', 'c1_f280', 'i5_f280']
lmax_for_plot = 5000

for spec in ['TT']:
    for so_map in so_maps:
        if 'f280' in so_map:
            ylims = {
                'TT': (-1000, 7000),
                'TE': (-200, 200),
                'TB': (-150, 150),
                'EE': (-10, 60),
                'EB': (-50, 50),
                'BB': (-50, 50)
            }
        else:
            ylims = {
                'TT': (-1000, 7000),
                'TE': (-200, 200),
                'TB': (-50, 50),
                'EE': (-10, 60),
                'EB': (-10, 10),
                'BB': (-10, 10)
            } 

        fig, ax = plt.subplots(figsize=(12, 6))

        mask = lb <= lmax_for_plot + 1000
        mask_th = l_th <= lmax_for_plot + 1000

        ax.semilogy(lb[mask], Dl_auto_1119[so_map][spec][mask], marker='.', markerfacecolor='white', markersize=10, color='C0', label=f'{so_map} x {so_map} (1119: $-$, 1019: --)')
        ax.semilogy(lb[mask], Dl_avg_1119[so_map][spec][mask], marker='.', markerfacecolor='white', markersize=10, color='C1', label=f'avg(ACT) x {so_map} (1119: $-$, 1019: --)')
        if spec != spec[::-1]:
            ax.semilogy(lb[mask], Dl_avg_1119[so_map][spec[::-1]][mask], marker='.', markerfacecolor='white', markersize=10, color='C2', label=f'{so_map} x avg(ACT) (1119: $-$, 1019: --)')

        ax.semilogy(lb[mask], Dl_auto_1019[so_map][spec][mask], marker='.', markerfacecolor='white', markersize=10, color='C0', linestyle='--')
        ax.semilogy(lb[mask], Dl_avg_1019[so_map][spec][mask], marker='.', markerfacecolor='white', markersize=10, color='C1', linestyle='--')
        if spec != spec[::-1]:
            ax.semilogy(lb[mask], Dl_avg_1019[so_map][spec[::-1]][mask], marker='.', markerfacecolor='white', markersize=10, color='C2', linestyle='--')

        ax.plot(l_th[mask_th], cmb_th[spec][mask_th], color='k', alpha=0.5, linestyle='--', zorder=0, label='LCDM (CMB-only)')
        ax.set_title(f'{so_map} {spec}', fontsize=16)
        ax.set_xlim(0, lmax_for_plot)
        # ax.set_ylim(ylims[spec])
        ax.set_xlabel(r'$\ell$', fontsize=16)
        ax.set_ylabel(fr'$D_\ell^{{{spec}}}\ \rm [\mu K^2]$', fontsize=16)
        ax.legend(fontsize=16)
        plt.show()

In [ ]:
so_maps = ['i1_f090', 'i3_f090', 'i4_f090', 'i6_f090', 'i1_f150', 'i3_f150', 'i4_f150', 'i6_f150', 'c1_f220', 'i5_f220', 'c1_f280', 'i5_f280']
lmax_for_plot = 3000

Dl_noise_1119 = {so_map: {} for so_map in so_maps}
Dl_noise_1019 = {so_map: {} for so_map in so_maps}
for so_map in so_maps:
    lb, Dl_noise_1119[so_map] = so_spectra.read_ps(d1119['spec_dir'] + f'Dl_so_{so_map}xso_{so_map}_noise.dat', spectra)
    lb, Dl_noise_1019[so_map] = so_spectra.read_ps(d1019['spec_dir'] + f'Dl_so_{so_map}xso_{so_map}_noise.dat', spectra)

for spec in ['TT', 'EE', 'BB']:
    for so_map in so_maps:
        if 'f280' in so_map:
            ylims = {
                'TT': (-1000, 7000),
                'TE': (-200, 200),
                'TB': (-150, 150),
                'EE': (-10, 60),
                'EB': (-50, 50),
                'BB': (-50, 50)
            }
        else:
            ylims = {
                'TT': (-1000, 7000),
                'TE': (-200, 200),
                'TB': (-50, 50),
                'EE': (-10, 60),
                'EB': (-10, 10),
                'BB': (-10, 10)
            } 

        fig, axs = plt.subplots(figsize=(12, 6), nrows=2, sharex=True, height_ratios=[4, 1])

        mask = lb <= lmax_for_plot + 1000
        mask_th = l_th <= lmax_for_plot + 1000

        l, bl = misc.read_beams(d1119[f"beam_T_so_{so_map}"], d1119[f"beam_pol_so_{so_map}"])
        _, blb = pspy_utils.naive_binning(l, bl['T'], d1119['binning_file'], d1119['lmax'])
        fac = 2 * np.pi / lb[mask]**2

        ax = axs[0]
        ax.semilogy(lb[mask], Dl_noise_1119[so_map][spec][mask] * fac * blb[mask]**2, marker='.', markerfacecolor='white', markersize=10, color='C0', label=f'{so_map} x {so_map} (1119: $-$, 1019: --)')
        ax.semilogy(lb[mask], Dl_noise_1019[so_map][spec][mask] * fac * blb[mask]**2, marker='.', markerfacecolor='white', markersize=10, color='C0', linestyle='--')
        # ax.plot(lb[mask], Dl_noise_1119[spec][mask] / Dl_noise_1019[spec][mask], marker='.', markerfacecolor='white', markersize=10, color='C0', label=f'{so_map} x {so_map} (1119: $-$, 1019: --)')
        
        ax.set_title(f'{so_map} {spec}', fontsize=16)
        ax.set_ylabel(fr'$N_\ell^{{{spec}}}\ \rm [\mu K^2]$', fontsize=16)
        ax.legend(fontsize=16)

        ax = axs[1]
        # ax.semilogy(lb[mask], Dl_noise_1119[spec][mask] * fac * blb[mask]**2, marker='.', markerfacecolor='white', markersize=10, color='C0', label=f'{so_map} x {so_map} (1119: $-$, 1019: --)')
        # ax.semilogy(lb[mask], Dl_noise_1019[spec][mask] * fac * blb[mask]**2, marker='.', markerfacecolor='white', markersize=10, color='C0', linestyle='--')
        ax.plot(lb[mask], Dl_noise_1119[so_map][spec][mask] / Dl_noise_1019[so_map][spec][mask], marker='.', markerfacecolor='white', markersize=10, color='C0', label=f'{so_map} x {so_map} (1119: $-$, 1019: --)')

        ax.grid()    
        ax.set_xlim(0, lmax_for_plot)
        ax.set_ylim(ymin=0, ymax=2)
        ax.set_xlabel(r'$\ell$', fontsize=16)
        ax.set_ylabel('ratio', fontsize=16)
        plt.show()